# 0.14 — Residual emergence graph (genAI benchmark)

Follow-up to [`0.13`](0.13-genai-graph-emergence.ipynb): cluster a **residual/emergence graph** (current structure minus baseline expectation), not the full weekly PPMI graph.

**Changes vs 0.13:**
- Residual edges `R = max(0, w − E[w])` + burst filter → Louvain on emergence graph only
- Share-normalized growth `G(C,t)` (not raw headline counts)
- Historical-similarity penalty vs baseline communities
- Reliability-capped score `S*`
- Contrastive top terms (interpretation only)

**Prerequisite:** `genai_graph_terms.parquet` from 0.13.

**Milestones (annotation only):** ChatGPT 2022-11-30 · CHAT ETF 2023-05-17

In [1]:
import re
from collections import Counter, defaultdict
from itertools import combinations
from math import log
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from networkx.algorithms.community import louvain_communities
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from tqdm.auto import tqdm

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = _ROOT / "notebooks" / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATE_START = pd.Timestamp("2021-01-01")
DATE_END = pd.Timestamp("2023-12-31")
BASELINE_END = pd.Timestamp("2022-10-31")
DISCOVERY_START = pd.Timestamp("2022-11-01")
DISCOVERY_END = pd.Timestamp("2022-12-31")
CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")
RANDOM_SEED = 42

FREQ = "W-MON"
MIN_DOC_FREQ = 5
WEEKLY_MIN_COUNT = 2
MAX_DOC_FREQ_PCT = 0.15
MIN_WEEK_HEADLINES = 500
MIN_PPMI = 0.0
LOUVAIN_RESOLUTION = 1.5
LOUVAIN_SEED = 42
ALPHA = 1.0
MIN_COMMUNITY_SIZE = 3
JACCARD_LINK = 0.25
MIN_HEADLINES = 5
EDGE_BURST_MIN = 0.5
TERM_BURST_MIN = 1.0
MIN_EDGE_COUNT = 1
RESIDUAL_MODE = "both"
TOP_TERMS = 12
TOP_HEADLINES = 10
TOP_RANK_PRINT = 15

TERMS_CACHE = OUTPUT_DIR / "genai_graph_terms.parquet"
RANKINGS_013 = OUTPUT_DIR / "genai_graph_emergence_rankings.parquet"
COMMUNITIES_PATH = OUTPUT_DIR / "genai_residual_communities.parquet"
RANKINGS_PATH = OUTPUT_DIR / "genai_residual_emergence_rankings.parquet"
EDGE_BURSTS_PATH = OUTPUT_DIR / "genai_residual_edge_bursts.parquet"
VALIDATION_PATH = OUTPUT_DIR / "genai_residual_validation.parquet"
COMPARE_PATH = OUTPUT_DIR / "genai_residual_compare_013.parquet"

FINANCE_STOP = set(ENGLISH_STOP_WORDS) | {
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock", "stocks",
    "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec",
    "sales", "deal", "chief", "cut", "raised", "raise", "buy", "sell", "net", "revenue",
    "beat", "miss", "forecast", "outlook", "business", "market", "markets", "global",
    "first", "second", "third", "fourth", "annual", "meeting", "plans", "plan", "bn",
    "march", "april", "june", "july", "august", "september", "october", "november", "december",
    "rated", "buy", "sell", "hold", "neutral", "perform", "outperform", "underperform",
    "overweight", "underweight", "equal-weight", "cut", "raise", "raised", "lowers", "upgrade",
    "downgrade", "maintains", "reiterates", "est", "eps", "adj", "sees", "expects", "forecast",
    "names", "appoints", "hires", "officer", "director", "chairman", "executive", "promotes",
    "tender", "offering", "offer", "notes", "bond", "bonds", "debt", "bills", "yield",
}
CUSTOM_STOP = {
    "inc", "plc", "ltd", "llc", "corp", "co", "sa", "ag", "nv", "group", "holdings",
    "ceo", "cfo", "says", "said", "new", "year", "today", "week", "day", "update",
    "report", "reports", "results", "announces", "announced", "shares", "stock",
    "stocks", "stake", "dividend", "q1", "q2", "q3", "q4", "fy", "unit", "mln", "bln", "pct",
}
TERM_STOP = FINANCE_STOP | CUSTOM_STOP

print(f"0.14 residual graph | baseline through {BASELINE_END.date()}")

0.14 residual graph | baseline through 2022-10-31


## 1. Load term cache from 0.13

In [2]:
if not TERMS_CACHE.exists():
    raise FileNotFoundError(f"Run 0.13 first — missing {TERMS_CACHE.name}")

news = pd.read_parquet(TERMS_CACHE)
news["date"] = pd.to_datetime(news["date"])
news["week"] = news["week"].astype(str)
if "hl_lc" not in news.columns:
    news["hl_lc"] = news["Headline"].str.lower()
print(f"Loaded {TERMS_CACHE.name}: {len(news):,} headlines")

Loaded genai_graph_terms.parquet: 2,954,113 headlines


## 2–4. Full graphs, baseline expectations, emergence graphs

In [3]:
def build_vocab(doc_terms: pd.Series, n_docs: int) -> set[str]:
    dfreq = Counter()
    for terms in doc_terms:
        for t in set(terms):
            dfreq[t] += 1
    max_df = int(MAX_DOC_FREQ_PCT * n_docs)
    return {t for t, c in dfreq.items() if MIN_DOC_FREQ <= c <= max_df}


def filter_terms(terms: list[str], vocab: set[str]) -> list[str]:
    out = []
    for t in terms:
        if t not in vocab:
            continue
        if any(tok in TERM_STOP for tok in t.split()):
            continue
        out.append(t)
    return sorted(out)


def build_ppmi_graph(rows: list[list[str]], min_count: int = WEEKLY_MIN_COUNT) -> nx.Graph:
    term_counts = Counter()
    pair_counts = Counter()
    n_documents = len(rows)
    if n_documents == 0:
        return nx.Graph()
    for phrases in rows:
        unique_terms = set(phrases)
        term_counts.update(unique_terms)
        pair_counts.update(combinations(sorted(unique_terms), 2))
    graph = nx.Graph()
    for term, count in term_counts.items():
        if count >= min_count:
            graph.add_node(term, count=count)
    for (term_a, term_b), pair_count in pair_counts.items():
        if term_a not in graph or term_b not in graph:
            continue
        p_ab = pair_count / n_documents
        p_a = term_counts[term_a] / n_documents
        p_b = term_counts[term_b] / n_documents
        ppmi = max(0.0, log(p_ab / (p_a * p_b)) if p_a * p_b > 0 else 0.0)
        if ppmi > MIN_PPMI:
            graph.add_edge(term_a, term_b, weight=ppmi, count=pair_count)
    return graph


def edge_key(a: str, b: str) -> tuple[str, str]:
    return (a, b) if a < b else (b, a)


def week_ts(week_str: str) -> pd.Timestamp:
    return pd.Period(week_str, freq=FREQ).start_time


def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 0.0
    return len(a & b) / len(a | b)


vocab = build_vocab(news["terms"], len(news))
print(f"Vocab: {len(vocab):,} terms")
news["terms_f"] = news["terms"].map(lambda ts: filter_terms(ts, vocab))

week_headline_counts = news.groupby("week").size().to_dict()
full_graphs: dict[str, nx.Graph] = {}
term_week_counts: dict[str, dict[str, int]] = defaultdict(lambda: defaultdict(int))
edge_week_counts: dict[tuple[str, str], dict[str, int]] = defaultdict(lambda: defaultdict(int))
edge_ppmi_week: dict[tuple[str, str], dict[str, float]] = defaultdict(lambda: defaultdict(float))

for week, grp in tqdm(news.groupby("week", sort=True), desc="full PPMI graphs"):
    rows = grp["terms_f"].tolist()
    if len(rows) < MIN_WEEK_HEADLINES:
        continue
    g = build_ppmi_graph(rows)
    if g.number_of_nodes() < MIN_COMMUNITY_SIZE:
        continue
    full_graphs[week] = g
    for terms in rows:
        for t in set(terms):
            term_week_counts[t][week] += 1
    for a, b, data in g.edges(data=True):
        k = edge_key(a, b)
        edge_week_counts[k][week] += data.get("count", 0)
        edge_ppmi_week[k][week] = data.get("weight", 0.0)

baseline_weeks = sorted([w for w in full_graphs if week_ts(w) <= BASELINE_END], key=week_ts)
print(f"Full graphs: {len(full_graphs)} weeks | baseline: {len(baseline_weeks)}")


def baseline_expectation(counter_by_week: dict, weeks: list[str]) -> float:
    if not weeks:
        return 0.0
    return float(np.mean([counter_by_week.get(w, 0) for w in weeks]))


def baseline_ppmi_expectation(k: tuple[str, str], weeks: list[str]) -> float:
    vals = [edge_ppmi_week[k][w] for w in weeks if edge_ppmi_week[k].get(w, 0) > 0]
    return float(np.mean(vals)) if vals else 0.0


E_term = {t: baseline_expectation(term_week_counts[t], baseline_weeks) for t in term_week_counts}
E_edge = {k: baseline_expectation(edge_week_counts[k], baseline_weeks) for k in edge_week_counts}
E_ppmi = {k: baseline_ppmi_expectation(k, baseline_weeks) for k in edge_week_counts}


def term_burst(term: str, week: str) -> float:
    obs = term_week_counts[term].get(week, 0)
    exp = E_term.get(term, 0.0)
    return log((obs + ALPHA) / (exp + ALPHA))


def edge_burst(a: str, b: str, week: str) -> float:
    k = edge_key(a, b)
    obs = edge_week_counts[k].get(week, 0)
    exp = E_edge.get(k, 0.0)
    return log((obs + ALPHA) / (exp + ALPHA))


def build_emergence_graph(full_g: nx.Graph, week: str) -> nx.Graph:
    emerge = nx.Graph()
    bursty_nodes = {n for n in full_g.nodes if term_burst(n, week) >= TERM_BURST_MIN}
    for a, b, data in full_g.edges(data=True):
        w = data.get("weight", 0.0)
        c = data.get("count", 0)
        k = edge_key(a, b)
        r_ppmi = max(0.0, w - E_ppmi.get(k, 0.0))
        r_count = max(0.0, c - E_edge.get(k, 0.0))
        residual_ok = (
            (RESIDUAL_MODE == "ppmi" and r_ppmi > 0)
            or (RESIDUAL_MODE == "count" and r_count > 0)
            or (RESIDUAL_MODE == "both" and (r_ppmi > 0 or r_count > 0))
        )
        eb = edge_burst(a, b, week)
        if not (residual_ok and c >= MIN_EDGE_COUNT and eb >= EDGE_BURST_MIN and w >= MIN_PPMI):
            continue
        emerge.add_edge(a, b, weight=max(r_ppmi, 0.01), count=c, r_ppmi=r_ppmi, r_count=r_count, edge_burst=eb)
    for n in bursty_nodes:
        if n not in emerge:
            emerge.add_node(n, bursty=True)
    return emerge


def detect_emergence_communities(emerge_g: nx.Graph) -> list[dict]:
    if emerge_g.number_of_nodes() == 0:
        return []
    comms = louvain_communities(
        emerge_g, weight="weight", resolution=LOUVAIN_RESOLUTION, seed=LOUVAIN_SEED,
    )
    out = []
    for idx, term_set in enumerate(comms):
        if len(term_set) < MIN_COMMUNITY_SIZE:
            continue
        terms = set(term_set)
        internal_edges = []
        for a, b in combinations(sorted(terms), 2):
            if emerge_g.has_edge(a, b):
                d = emerge_g[a][b]
                internal_edges.append((a, b, d.get("count", 0)))
        out.append({
            "community_id": idx,
            "terms": terms,
            "internal_edges": internal_edges,
            "n_terms": len(terms),
        })
    return out


emergence_graphs: dict[str, nx.Graph] = {}
communities_by_week: dict[str, list[dict]] = {}
for week in tqdm(sorted(full_graphs.keys(), key=week_ts), desc="emergence graphs"):
    eg = build_emergence_graph(full_graphs[week], week)
    emergence_graphs[week] = eg
    communities_by_week[week] = detect_emergence_communities(eg)

print(f"Emergence graphs: {len(emergence_graphs)} | "
      f"avg communities/week: {np.mean([len(communities_by_week[w]) for w in communities_by_week]):.1f}")

Vocab: 201,787 terms


full PPMI graphs:   0%|          | 0/157 [00:00<?, ?it/s]

Full graphs: 157 weeks | baseline: 96


emergence graphs:   0%|          | 0/157 [00:00<?, ?it/s]

Emergence graphs: 157 | avg communities/week: 65.1


## 5–7. Tracking, baseline library, features, S* ranking

In [4]:
def coherence_ext(full_g: nx.Graph, terms: set[str]) -> float:
    internal, external = 0.0, 0.0
    for a in terms:
        if a not in full_g:
            continue
        for nbr in full_g.neighbors(a):
            w = full_g[a][nbr].get("weight", 0.0)
            if nbr in terms:
                internal += w
            else:
                external += w
    return internal / (internal + external + 1e-9)


def contrast_top_terms(terms: set[str], week: str, community_indices: list[int]) -> list[str]:
    if not terms:
        return []
    in_ctr = Counter()
    out_ctr = Counter()
    grp = news.loc[news.week == week]
    comm_set = set(community_indices)
    for idx, tlist in zip(grp.index, grp["terms_f"]):
        ts = set(tlist)
        tgt = in_ctr if idx in comm_set else out_ctr
        for t in ts & terms:
            tgt[t] += 1
    n_in, n_out = max(len(comm_set), 1), max(len(grp) - len(comm_set), 1)
    scored = []
    for t in terms:
        p_in = (in_ctr[t] + ALPHA) / (n_in + ALPHA)
        p_out = (out_ctr[t] + ALPHA) / (n_out + ALPHA)
        scored.append((log(p_in / p_out), t))
    scored.sort(reverse=True)
    return [t for _, t in scored[:TOP_TERMS]]


# --- baseline community library (emergence graph, baseline weeks) ---
baseline_communities: list[set[str]] = []
for w in baseline_weeks:
    for c in communities_by_week.get(w, []):
        baseline_communities.append(c["terms"])
print(f"Baseline community library: {len(baseline_communities)} communities")


def hist_sim(terms: set[str]) -> float:
    if not baseline_communities:
        return 0.0
    return max(jaccard(terms, bc) for bc in baseline_communities)


# --- track communities across weeks ---
next_track = 0
prev_comms: list[dict] = []
track_records: list[dict] = []
track_baseline_shares: dict[int, list[float]] = defaultdict(list)

for week in sorted(communities_by_week.keys(), key=week_ts):
    grp = news.loc[news.week == week]
    hl_idx = grp.index.tolist()
    hl_terms = grp["terms_f"].tolist()
    full_g = full_graphs[week]
    emerge_g = emergence_graphs[week]
    matched_prev: set[int] = set()
    week_comms: list[dict] = []

    for c in communities_by_week[week]:
        best_j, best_track = 0.0, None
        for pc in prev_comms:
            if pc["track_id"] in matched_prev:
                continue
            j = jaccard(c["terms"], pc["terms"])
            if j >= JACCARD_LINK and j > best_j:
                best_j, best_track = j, pc["track_id"]
        if best_track is None:
            best_track = next_track
            next_track += 1
        else:
            matched_prev.add(best_track)

        members = [i for i, terms in zip(hl_idx, hl_terms) if c["terms"] & set(terms)]
        rec = dict(c)
        rec.update({
            "track_id": best_track,
            "period": week,
            "headline_count": len(members),
            "headline_indices": members,
        })

        tb = [term_burst(t, week) for t in rec["terms"]]
        eb = [edge_burst(a, b, week) for a, b, _ in rec["internal_edges"]]
        rec["mean_term_burst"] = float(np.mean(tb)) if tb else 0.0
        rec["mean_edge_burst"] = float(np.mean(eb)) if eb else 0.0
        rec["vocabulary_novelty"] = float(np.mean([
            1.0 if E_term.get(t, 0) < 0.5 else 0.0 for t in rec["terms"]
        ]))
        rec["edge_novelty"] = float(np.mean([
            1.0 if E_edge.get(edge_key(a, b), 0) < 0.5 else 0.0
            for a, b, _ in rec["internal_edges"]
        ])) if rec["internal_edges"] else 0.0
        rec["coherence_ext"] = coherence_ext(full_g, rec["terms"])
        rec["hist_sim"] = hist_sim(rec["terms"])
        rec["top_terms"] = contrast_top_terms(rec["terms"], week, members)

        week_total = week_headline_counts[week]
        share = rec["headline_count"] / week_total if week_total else 0.0
        rec["share"] = share
        if week_ts(week) <= BASELINE_END:
            track_baseline_shares[best_track].append(share)

        week_comms.append(rec)
        track_records.append(rec)
    prev_comms = week_comms

# share growth + persistence
min_share_prior = 1.0 / max(week_headline_counts.values())
track_weeks: dict[int, list[str]] = defaultdict(list)
for c in track_records:
    track_weeks[c["track_id"]].append(c["period"])

for c in track_records:
    tid = c["track_id"]
    base_shares = track_baseline_shares.get(tid, [])
    p_bar = float(np.mean(base_shares)) if base_shares else min_share_prior
    c["share_growth"] = log((c["share"] + ALPHA) / (p_bar + ALPHA))
    ws = sorted(track_weeks[tid], key=week_ts)
    streak = 1
    for i in range(1, len(ws)):
        if (week_ts(ws[i]) - week_ts(ws[i - 1])).days <= 8:
            streak += 1
        else:
            break
    c["persistence"] = streak
    c["first_seen"] = ws[0]

print(f"Track records: {len(track_records):,} | tracks: {next_track:,}")

Baseline community library: 5528 communities
Track records: 10,222 | tracks: 10,210


## 8–9. S / S* ranking + representative headlines

In [5]:
POS_FEATURES = [
    "share_growth", "mean_term_burst", "mean_edge_burst",
    "vocabulary_novelty", "edge_novelty", "coherence_ext", "persistence",
]


def zscore_series(s: pd.Series) -> pd.Series:
    if s.std(ddof=0) == 0 or len(s) < 2:
        return pd.Series(0.0, index=s.index)
    return (s - s.mean()) / s.std(ddof=0)


rows = []
for c in track_records:
    rows.append({
        "period": c["period"], "community_id": c["community_id"], "track_id": c["track_id"],
        "top_terms": ", ".join(c["top_terms"]), "headline_count": c["headline_count"],
        "share": c["share"], "first_seen": c["first_seen"], "hist_sim": c["hist_sim"],
        **{f: c[f] for f in POS_FEATURES},
    })
rank_df = pd.DataFrame(rows)
rank_df["S"] = 0.0
rank_df["S_star"] = 0.0
reliability_denom = log(1 + MIN_HEADLINES)

for week, sub in rank_df.groupby("period"):
    z_pos = np.vstack([zscore_series(sub[f]) for f in POS_FEATURES])
    z_hist = zscore_series(sub["hist_sim"])
    s = np.mean(z_pos, axis=0) - z_hist.values
    rel = np.minimum(1.0, np.log1p(sub["headline_count"].values) / reliability_denom)
    rank_df.loc[sub.index, "S"] = s
    rank_df.loc[sub.index, "S_star"] = s * rel

rank_df["rank_in_week"] = rank_df.groupby("period")["S_star"].rank(ascending=False, method="first").astype(int)
rank_df = rank_df.sort_values(["period", "rank_in_week"]).reset_index(drop=True)
rank_df.to_parquet(RANKINGS_PATH, index=False)
print(f"Wrote {RANKINGS_PATH.name}: {len(rank_df):,} rows")

comm_df = pd.DataFrame([{
    "period": c["period"], "community_id": c["community_id"], "track_id": c["track_id"],
    "top_terms": ", ".join(c["top_terms"]), "headline_count": c["headline_count"],
    "share": c["share"], "share_growth": c["share_growth"], "hist_sim": c["hist_sim"],
    "coherence_ext": c["coherence_ext"], "mean_term_burst": c["mean_term_burst"],
    "mean_edge_burst": c["mean_edge_burst"], "persistence": c["persistence"],
    "first_seen": c["first_seen"],
} for c in track_records])
comm_df.to_parquet(COMMUNITIES_PATH, index=False)

comm_lookup = {(c["period"], c["track_id"]): c for c in track_records}
rep_rows = []
disc = rank_df.loc[rank_df.period.map(week_ts).between(DISCOVERY_START, DISCOVERY_END)]
for _, r in disc[disc.rank_in_week <= TOP_RANK_PRINT].iterrows():
    c = comm_lookup.get((r["period"], r["track_id"]))
    if c is None:
        continue
    emerge_g = emergence_graphs.get(r["period"])
    term_w = {}
    if emerge_g:
        for t in c["terms"]:
            term_w[t] = (emerge_g.degree(t, weight="weight") if t in emerge_g else 0) * np.exp(term_burst(t, r["period"]))
    scores = []
    for idx in c["headline_indices"][:5000]:
        score = sum(term_w.get(t, 0) for t in set(news.at[idx, "terms_f"]) & c["terms"])
        scores.append((score, news.at[idx, "Headline"]))
    scores.sort(key=lambda x: -x[0])
    for rep_rank, (score, hl) in enumerate(scores[:TOP_HEADLINES], 1):
        rep_rows.append({
            "period": r["period"], "track_id": r["track_id"], "rank_in_week": int(r["rank_in_week"]),
            "rep_rank": rep_rank, "score": score, "headline": hl[:240],
        })
rep_df = pd.DataFrame(rep_rows)

print("\nTop S* communities — discovery window:")
for week in sorted(disc["period"].unique(), key=week_ts):
    sub = rank_df[rank_df.period == week].head(5)
    print(f"\n  {week}")
    for _, r in sub.iterrows():
        print(f"    #{int(r.rank_in_week)} track {int(r.track_id)} S*={r.S_star:.2f} hist_sim={r.hist_sim:.2f}  {r.top_terms[:70]}")

Wrote genai_residual_emergence_rankings.parquet: 10,222 rows

Top S* communities — discovery window:

  2022-11-01/2022-11-07
    #1 track 5595 S*=1.92 hist_sim=0.02  tax, focus, german, boss, billionaire, bankers, restaurant, shut, shut
    #2 track 5531 S*=1.44 hist_sim=0.02  apple, iphone, shipments, rogers, dupont, termination, albertsons, tri
    #3 track 5573 S*=1.35 hist_sim=0.02  exports, blue, petroleum, windfall, resigns, ton, table, tourism, fosu
    #4 track 5560 S*=1.21 hist_sim=0.02  twitter, musk, jobs, technology, time, general, pfizer, dong, half, ad
    #5 track 5551 S*=1.07 hist_sim=0.02  riyals, saudi, realty, electric, east, sign, build, core ffo, office, 

  2022-11-08/2022-11-14
    #1 track 5668 S*=1.08 hist_sim=0.02  ftx, crypto, tech, seeks, assets, seen, funds, com, speaks, digital, r
    #2 track 5656 S*=0.83 hist_sim=0.02  president, boe, pacific, coo, step, cathay, john, changes, occidental,
    #3 track 5612 S*=0.68 hist_sim=0.03  open, approval, holders,

## 10–11. Validation, 0.13 comparison, visualizations

In [6]:
GENAI_T1 = [
    r"generative ai", r"generative artificial intelligence",
    r"large language model", r"large language models", r"\bllms\b",
    r"chatgpt", r"gpt-3\.5", r"gpt-3", r"gpt-4",
    r"foundation model", r"foundation models",
    r"stable diffusion", r"midjourney", r"dall-e", r"dalle",
]
GENAI_T2 = [
    r"\bopenai\b", r"\banthropic\b", r"chatgpt-like", r"chatgpt-style",
    r"prompt engineering", r"ai chatbot", r"ai chat bot",
]
GENAI_STANDARD = re.compile("|".join(f"(?:{p})" for p in GENAI_T1 + GENAI_T2), re.I)
GENAI_TERM_HINT = re.compile(
    r"chatgpt|openai|chatbot|generative|gpt-3|gpt-4|anthropic|copilot|bard|gemini|llm", re.I,
)


def pct_genai(headlines: list[str]) -> float:
    if not headlines:
        return 0.0
    return round(pd.Series(headlines).str.contains(GENAI_STANDARD, na=False).mean() * 100, 1)


val_rows = []
for c in track_records:
    if not (DISCOVERY_START <= week_ts(c["period"]) <= DISCOVERY_END):
        continue
    hls = [news.at[i, "Headline"] for i in c["headline_indices"]]
    rep_hls = rep_df.loc[
        (rep_df.period == c["period"]) & (rep_df.track_id == c["track_id"]), "headline"
    ].tolist() if len(rep_df) else []
    rk = rank_df.loc[(rank_df.period == c["period"]) & (rank_df.track_id == c["track_id"])].iloc[0]
    val_rows.append({
        "period": c["period"], "track_id": c["track_id"],
        "rank_in_week": int(rk["rank_in_week"]), "S_star": float(rk["S_star"]),
        "hist_sim": float(c["hist_sim"]),
        "pct_standard_all": pct_genai(hls), "pct_standard_rep": pct_genai(rep_hls),
        "top_terms": ", ".join(c["top_terms"]), "first_seen": c["first_seen"],
    })
val_df = pd.DataFrame(val_rows).sort_values(["period", "rank_in_week"])
val_df.to_parquet(VALIDATION_PATH, index=False)

best = val_df.sort_values(["pct_standard_rep", "pct_standard_all", "S_star"], ascending=False).head(10)
print("Best genAI overlap (0.14):")
print(best[["period", "rank_in_week", "pct_standard_rep", "S_star", "hist_sim", "top_terms"]].to_string(index=False))

diag = []
for c in track_records:
    if not (DISCOVERY_START <= week_ts(c["period"]) <= DISCOVERY_END):
        continue
    hits = [t for t in c["terms"] if GENAI_TERM_HINT.search(t)]
    if not hits:
        continue
    rk = rank_df.loc[(rank_df.period == c["period"]) & (rank_df.track_id == c["track_id"])].iloc[0]
    diag.append({
        "period": c["period"], "track_id": c["track_id"], "rank_in_week": int(rk["rank_in_week"]),
        "S_star": float(rk["S_star"]), "genai_terms": ", ".join(sorted(hits)[:8]),
        "top_terms": ", ".join(c["top_terms"])[:80],
    })
if diag:
    print("\nGenAI-vocabulary communities (diagnostic):")
    print(pd.DataFrame(diag).sort_values(["period", "rank_in_week"]).to_string(index=False))

# edge burst table with residuals
edge_rows = []
for week in sorted(full_graphs.keys(), key=week_ts):
    if not (DISCOVERY_START <= week_ts(week) <= DISCOVERY_END):
        continue
    for a, b, data in full_graphs[week].edges(data=True):
        k = edge_key(a, b)
        eb = edge_burst(a, b, week)
        if eb <= 0:
            continue
        edge_rows.append({
            "period": week, "term_a": a, "term_b": b,
            "baseline_co": E_edge.get(k, 0), "current_co": edge_week_counts[k].get(week, 0),
            "baseline_ppmi": E_ppmi.get(k, 0), "current_ppmi": data.get("weight", 0),
            "residual_ppmi": max(0, data.get("weight", 0) - E_ppmi.get(k, 0)),
            "edge_burst": eb,
        })
edge_df = pd.DataFrame(edge_rows).sort_values("edge_burst", ascending=False)
edge_df.to_parquet(EDGE_BURSTS_PATH, index=False)
print(f"\nTop residual edge bursts (Nov–Dec):")
print(edge_df.head(12).to_string(index=False))

# 0.13 comparison
compare_rows = []
v013 = OUTPUT_DIR / "genai_graph_validation.parquet"
val13 = pd.read_parquet(v013) if v013.exists() else pd.DataFrame()
if RANKINGS_013.exists():
    r13 = pd.read_parquet(RANKINGS_013)
    r13_disc = r13[r13.period.map(week_ts).between(DISCOVERY_START, DISCOVERY_END)]
    for week in sorted(r13_disc["period"].unique(), key=week_ts):
        sub14 = val_df[val_df.period == week].sort_values("pct_standard_rep", ascending=False)
        sub13_val = val13[val13.period == week].sort_values("pct_standard_rep", ascending=False) if len(val13) else pd.DataFrame()
        best14 = sub14.iloc[0] if len(sub14) else None
        best13 = sub13_val.iloc[0] if len(sub13_val) else None
        compare_rows.append({
            "period": week,
            "best_013_rank": int(best13["rank_in_week"]) if best13 is not None else None,
            "best_013_pct_rep": best13["pct_standard_rep"] if best13 is not None else None,
            "best_014_rank": int(best14["rank_in_week"]) if best14 is not None else None,
            "best_014_pct_rep": best14["pct_standard_rep"] if best14 is not None else None,
            "best_014_S_star": best14["S_star"] if best14 is not None else None,
            "top_014_terms": best14["top_terms"][:80] if best14 is not None else "",
        })
    # overall bests
    best13_rep = pd.read_parquet(v013)["pct_standard_rep"].max() if v013.exists() else None
    best14_rep = val_df["pct_standard_rep"].max() if len(val_df) else 0
    summary = pd.DataFrame([{
        "metric": "best_pct_standard_rep", "value_013": best13_rep, "value_014": best14_rep,
    }])
    pd.DataFrame(compare_rows).to_parquet(COMPARE_PATH, index=False)
    print(f"\n0.13 vs 0.14 — best rep overlap: {best13_rep}% vs {best14_rep}%")
    print(f"Wrote {COMPARE_PATH.name}")
else:
    print("Skip 0.13 comparison — rankings parquet missing")

# timeline
top_tracks = (
    rank_df.loc[rank_df.period.map(week_ts).between(DISCOVERY_START, DISCOVERY_END)]
    .groupby("track_id")["S_star"].max().sort_values(ascending=False).head(10).index.tolist()
)
fig = go.Figure()
for tid in top_tracks:
    sub = rank_df[rank_df.track_id == tid].sort_values("period")
    fig.add_trace(go.Scatter(
        x=sub["period"].map(week_ts), y=sub["S_star"],
        mode="lines+markers", name=f"track {tid}",
    ))
fig.add_vline(x=CHATGPT_LAUNCH, line_dash="dash", line_color="gray")
fig.update_layout(title="Top 10 S* tracks (residual graph)", height=480)
fig.write_html(OUTPUT_DIR / "genai_residual_emergence_timeline.html")

# emergence network for best genAI diagnostic week
if diag:
    d0 = sorted(diag, key=lambda x: (x["rank_in_week"], -x["S_star"]))[0]
    pw = d0["period"]
    if pw in emergence_graphs:
        g = emergence_graphs[pw]
        if g.number_of_nodes() > 0:
            pos = nx.spring_layout(g, seed=RANDOM_SEED, k=0.6)
            edge_x, edge_y = [], []
            for a, b in g.edges():
                x0, y0 = pos[a]; x1, y1 = pos[b]
                edge_x += [x0, x1, None]; edge_y += [y0, y1, None]
            fig2 = go.Figure()
            fig2.add_trace(go.Scatter(x=edge_x, y=edge_y, mode="lines", line=dict(width=0.5, color="#888"), hoverinfo="none"))
            fig2.add_trace(go.Scatter(
                x=[pos[n][0] for n in g.nodes()], y=[pos[n][1] for n in g.nodes()],
                mode="markers+text", text=list(g.nodes()), textposition="top center", marker=dict(size=8),
            ))
            fig2.update_layout(title=f"Emergence graph — {pw}", showlegend=False, height=600)
            fig2.write_html(OUTPUT_DIR / "genai_residual_network_W48.html")
            print(f"Wrote genai_residual_network_W48.html ({g.number_of_nodes()} nodes)")

Best genAI overlap (0.14):
               period  rank_in_week  pct_standard_rep    S_star  hist_sim                                                                                                 top_terms
2022-12-06/2022-12-12             9               0.0  0.454592  0.022321            drops, short, frn, premarket, seazen, google, acquisitions, wrong, info, exp, sellers, sec frn
2022-12-13/2022-12-19            30               0.0  0.253241  0.023256              profit, malaysia, fixes, delta, shell, erdogan, resume, weaker, rival, jobs, curb, elections
2022-11-29/2022-12-05            70               0.0 -0.461363  0.025000     united, bofa, downgrades, slides, systems, quarter, eisai, internet, debate, glass, valuation, upside
2022-12-06/2022-12-12            25               0.0  0.269921  0.025510          contract, ytd, partners, trader, export, mobile, talk, levels, dubai, regulatory, nippon, moving
2022-12-20/2022-12-26            25               0.0  0.573916  0.029557

## Quick reload

In [ ]:
for path in [RANKINGS_PATH, VALIDATION_PATH, EDGE_BURSTS_PATH, COMMUNITIES_PATH, COMPARE_PATH]:
    if path.exists():
        t = pd.read_parquet(path)
        print(f"{path.name}: {len(t):,} rows")